### Bronze ETL LGS
This notebook setups data ingestion from both JDBC and Azure Delta Data Lake external storage to the bronze layer


In [0]:
%sql
-- Setup catalog if not exists
-- CREATE CATALOG IF NOT EXISTS jarvis_training;
-- Setup schemas for medallion architecture, you can also use the GUI
-- Schema == Database
CREATE SCHEMA IF NOT EXISTS jarvis_training.bronze;
CREATE EXTERNAL VOLUME IF NOT EXISTS jarvis_training.bronze.raw_data
LOCATION 'abfss://raw@jarvistrainingadls.dfs.core.windows.net/';CREATE SCHEMA IF NOT EXISTS jarvis_training.silver;
CREATE SCHEMA IF NOT EXISTS jarvis_training.gold;

In [0]:
url = (
    "jdbc:sqlserver://jarvis-training-sql-canada.database.windows.net:1433;"
    "database=jarvis-training;"
)

transaction_data_df = (spark.read
  .format("jdbc")
  .option("url", url)
  .option("dbtable", "dbo.transactions_data")
  .option("user", "jason121301")
  .option("password", "Jarvis1234")
  .load()
)

cards_data_df = (spark.read
  .format("jdbc")
  .option("url", url)
  .option("dbtable", "dbo.cards_data")
  .option("user", "jason121301")
  .option("password", "Jarvis1234")
  .load()
)

transaction_data_df.head() # display your dataframe
cards_data_df.head()

In [0]:
rom pyspark.sql.types import StructType, StructField, MapType, StringType
from pyspark.sql.functions import from_json, col, explode, regexp_extract, regexp_replace, trim
import numpy as np

# Read the JSON file from volume
users_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/jarvis_training/bronze/raw_data/csv/users_data.csv")
)


fraud_schema = StructType([
    StructField(
        "target",
        MapType(StringType(), StringType()),
        True
    )
])


fraud_df = (
    spark.read
    .schema(fraud_schema)
    .option("multiLine", "true")
    .json("/Volumes/jarvis_training/bronze/raw_data/json/train_fraud_labels.json")
)

fraud_exploded_df = (
    fraud_df
    .select(explode("target").alias("transaction_id", "fraud_label"))
)

fraud_exploded_df.head()

raw_text_df = spark.read.text(
    "/Volumes/jarvis_training/bronze/raw_data/json/mcc_codes.json"
)

parsed_df = raw_text_df.select(
    regexp_extract(col("value"), r'"(\d{4})"', 1).alias("mcc_code"),
    regexp_extract(col("value"), r':\s*"([^"]+)"', 1).alias("mcc_description")
)

display(parsed_df)



In [0]:
transaction_data_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.transaction")

cards_data_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.cards")

users_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.users")

parsed_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.mcc_codes")

fraud_exploded_df.write.mode("overwrite").saveAsTable("jarvis_training.bronze.fraud_labels")